# Distributed Joins and Complex Aggregations

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 05_joins_and_aggregations.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Complete reference on distributed join types (inner, left, right, full, cross, semi, anti) and advanced aggregations (cube, rollup, pivot).
- **Why does this topic exist?**: Joins and aggregations are the most compute- and network-intensive operations in distributed data engineering.
- **What problem does it solve?**: Incorrect join types lead to duplicate rows, dropped data, or massive Cartesian products.
- **Where is it used in real-world Data Engineering?**: Denormalizing star-schema fact and dimension tables in data warehousing.

---

## 2. Distributed Join Types

```
+-----------------------------------------------------------------------+
| JOIN TYPES IN SPARK                                                   |
+-------------------+---------------------------------------------------+
| Inner Join        | Keeps matching rows in both tables.               |
| Left / Right Join | Keeps all rows from left/right + matching right/left|
| Full Outer Join   | Keeps all rows from both tables, filling nulls.   |
| Left Semi Join    | Filters left rows that have a match in right.     |
|                   | (Columns from right are NOT added).               |
| Left Anti Join    | Filters left rows that have NO match in right.    |
| Cross Join        | Cartesian product (M x N rows) - USE CAUTION!     |
+-------------------+---------------------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Joins_Aggregations_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Hands-On: Three-Table Join (Orders + Customers + Products)

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
customers = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/customers.csv")
products = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/products.csv")

# Join orders with customers (left join) and products (left join)
enriched_orders = (
    orders
    .join(customers, on="customer_id", how="left")
    .join(products, on="product_id", how="left")
    .select(
        "order_id", "order_ts", "customer_id", "first_name", "country", "region",
        "product_id", "product_name", "category", "quantity", "unit_price", "status"
    )
)

enriched_orders.show(3)

## 4. Left Anti Join: Finding Orphan Customers

Find customers who have never placed an order using Left Anti Join.

In [ ]:
# Customers with ZERO orders
dormant_customers = (
    customers
    .join(orders, on="customer_id", how="left_anti")
    .select("customer_id", "first_name", "email", "signup_date")
)

print(f"Total customers: {customers.count()}")
print(f"Dormant customers with 0 orders: {dormant_customers.count()}")
dormant_customers.show(3)

## 5. Advanced Aggregations: `rollup`, `cube`, and `pivot`

In [ ]:
# Pivot: Total sales by Region across Product Categories
pivot_df = (
    enriched_orders
    .filter("status = 'Completed'")
    .groupBy("region")
    .pivot("category")
    .agg(F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2))
    .na.fill(0.0)
)

print("Sales Pivot Table (Region vs Category):")
pivot_df.show()

---

## Experiment: Left Semi Join vs Inner Join Row Count and Performance

- **1. What are we testing?**: Prove that Left Semi Join does not duplicate left table rows even if the right table has 1-to-many matches.
- **2. What do we expect?**: Left Semi Join returns at most 1 copy of left rows and does not project right table columns.

In [ ]:
# Left semi join: find customers who placed at least 1 order
semi_df = customers.join(orders, on="customer_id", how="left_semi")

print(f"Total customers:                 {customers.count()}")
print(f"Customers with >= 1 order (semi): {semi_df.count()}")
print("Columns in semi join result:", semi_df.columns)

**3. What happened?**: semi_df contains only customers columns; no duplicates occurred despite customers having multiple orders.  
**4. Why did it happen?**: Left semi join tests existence of a match; it stops after finding the first match and outputs the left row.  
**5. Practical Engineering Lesson**: Use left_semi instead of inner join + distinct when filtering a table based on existence in another table.

---

## Practical Exercises

### Exercise 1 (Beginner): Multi-Column Aggregations with agg()
**Problem Statement**: Compute total revenue, avg unit price, and unique order count per product category.  
**Input / Context**: enriched_orders

In [ ]:
# Exercise 1: Category metrics

### Exercise 2 (Intermediate): Rollup for Hierarchical Subtotals
**Problem Statement**: Use rollup('region','channel') to compute subtotals at the region level and overall grand total.  
**Input / Context**: enriched_orders

In [ ]:
# Exercise 2: Rollup subtotals

### Exercise 3 (Advanced): Detect Products with Zero Sales
**Problem Statement**: Use a Left Anti Join between products and orders to identify all products that have never been purchased.  
**Input / Context**: products and orders

In [ ]:
# Exercise 3: Products with zero orders

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Multi-Column Aggregations with agg()):
```python
enriched_orders.groupBy("category").agg(
    F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("total_rev"),
    F.round(F.avg("unit_price"), 2).alias("avg_price"),
    F.countDistinct("order_id").alias("unique_orders")
).show()
```

#### Solution 2 (Rollup for Hierarchical Subtotals):
```python
enriched_orders.rollup("region","channel").agg(
    F.count("order_id").alias("orders_count"),
    F.round(F.sum(F.col("quantity")*F.col("unit_price")), 2).alias("revenue")
).orderBy(F.col("region").asc_nulls_last(), F.col("channel").asc_nulls_last()).show(10)
```

#### Solution 3 (Detect Products with Zero Sales):
```python
unsold_products = products.join(orders, on="product_id", how="left_anti")
print("Unsold products count:", unsold_products.count())
unsold_products.select("product_id","product_name","category").show(5)
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Join Types**: Know when to use inner, left, left_semi, and left_anti.
- **Left Semi Efficiency**: Filters for existence without duplicating rows or pulling right columns.
- **Pivot & Rollup**: Powerful multidimensional aggregation primitives built into Spark SQL.

### Essential Technical Terms
- `Left Semi Join`: Returns left rows matching right table without right columns.
- `Left Anti Join`: Returns left rows with NO match in right table.
- `Pivot`: Rotates distinct column values into horizontal column headers.

### Common Engineering Mistakes
- **Unintentional Cross Joins**: Missing join conditions generate massive Cartesian products.
- **Duplicate Column Names**: Joining two tables with matching column names without disambiguation causes ambiguous reference errors.

### Interview Questions on This Topic
- **Q: What is the difference between Left Semi and Inner Join?**
  - *A*: Left semi only checks for existence, does not duplicate left rows on 1:N matches, and does not include right table columns.
- **Q: How do you resolve ambiguous column names after a join?**
  - *A*: Drop the column prior to join, or join on a string name ('id') rather than expression (df1.id == df2.id).

### What to Study Next
Open 03_dataframe_and_spark_sql/06_window_functions.ipynb to master analytical window functions.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')